In [0]:
library(dataiku)
library(stats) # need this to calculate Mahalanobis Distance
library(parallel) # parallelize
library(dplyr)
library(FNN)
library(cluster)

In [0]:
# Recipe inputs
counterfactual_test_data <- dkuReadDataset("counterfactual_test_data", samplingMethod="head", nbRows=100000)
hurdle_predictions_testing <- dkuManagedFolderPath("5NPBmWH1")

In [0]:
colnames(counterfactual_test_data)

In [0]:
# get unique municipality observations
mun_properties  <- counterfactual_test_data %>% 
    distinct(Mun_Code, 
             blue_ss_frac,
             blue_ls_frac,
             red_ls_frac,
             orange_ls_frac,
             yellow_ss_frac,
             red_ss_frac,
             orange_ss_frac,
             yellow_ls_frac,
             roof_strong_wall_strong,
             roof_strong_wall_light,
             roof_strong_wall_salv,
             roof_light_wall_strong,
             roof_light_wall_light,
             roof_light_wall_salv,
             roof_salv_wall_strong,
             roof_salv_wall_light,
             roof_salv_wall_salv,
             island_groups,
             .keep_all = FALSE)

In [0]:
# variables I'm interested in for matching:
match_vars  <- c('blue_ss_frac',
                    'blue_ls_frac',
                    'red_ls_frac',
                    'orange_ls_frac',
                    'yellow_ss_frac',
                    'red_ss_frac',
                    'orange_ss_frac',
                    'yellow_ls_frac',
                    'roof_strong_wall_strong',
                    'roof_strong_wall_light',
                    'roof_strong_wall_salv',
                    'roof_light_wall_strong',
                    'roof_light_wall_light',
                    'roof_light_wall_salv',
                    'roof_salv_wall_strong',
                    'roof_salv_wall_light',
                    'roof_salv_wall_salv'
                   )

In [0]:
# Normalize the variables using z-score
mun_scaled <- mun_properties %>%
  mutate(across(c(blue_ss_frac:roof_salv_wall_salv), scale))

In [0]:
nrow(matched_df)

In [0]:
# Find the minimum Mahalanobis distances for each observation
min_distances <- apply(mah_dist, 1, min)

# Define a threshold for the Mahalanobis distance (adjust this as needed)
threshold <- 0.00000005  # Example threshold

# Filter the observations that have a distance below the threshold
valid_matches <- min_distances < threshold

# Subset the matched dataset to only include those observations with close matches
matched_subset <- mun_properties[valid_matches, ]

head(matched_subset)

In [0]:
nrow(matched_subset)

In [0]:
# Recipe outputs
matching_counterfactuals <- dkuManagedFolderPath("ZO3oPxC1")